In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
from mpl_toolkits.axes_grid1 import make_axes_locatable, axes_size as maxes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib as mpl
import matplotlib.axes as maxes
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
import cartopy.crs as ccrs
from scipy.stats import pearsonr
from matplotlib.gridspec import GridSpec

plot_path = '/home/u/u241308/figures/'

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
import numpy as np

In [ ]:
#anomaly reference period
ref_min = 1985
ref_max = 2014

In [ ]:
time_min = 1960
time_max = 2020

# Select lead years

In [ ]:
lead_year = 1 #2

In [ ]:
#1-13 for Dec-Nov first lead year, 13-25 for Dec-Nov second lead year
if lead_year == 1:
    lead_min = 0 # 2 to start in January, first lead year
    lead_max = 12 # 14 To end in December, first lead year
if lead_year == 2:
    lead_min = 12 # 14 to start in January, second lead year
    lead_max = 24 # 26 to end in December, second lead year

# Hybrid ML

In [ ]:
# Load, slice, and collect datasets
sliced_datasets1 = []
sliced_datasets2 = []
for y_id in range(time_min,time_max):
    path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/sst_ML_NA/anomaly/'
    file = 'sst_%i_r1-16i2p3-LR_26_months_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path+file) as sst_ML:
        sst_ML_ly1 = sst_ML.tos.isel(time=slice(lead_min,lead_max))
    sliced_datasets1.append(sst_ML_ly1.mean(dim='time').expand_dims(predicted_year=[y_id]))
    
#concatenate along time dimension
sst_ML_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
sst_ML_ly1['lon'] = np.where(sst_ML_ly1.lon > 180,sst_ML_ly1.lon-360,sst_ML_ly1.lon)
ind_sort = np.argsort(sst_ML_ly1.lon)
sst_ML_ly1 = sst_ML_ly1[:,:,:,ind_sort.values]

#change 'depth' dimension name to 'sfc', so the dimension name matches with z500 (only for ML, not for MPI)
sst_ML_ly1 = sst_ML_ly1.rename({"depth": "sfc"})
#give different coordinates for each ensemble member in sfc:
sst_ML_ly1 = sst_ML_ly1.assign_coords(sfc=np.arange(1, 17))

#ensemble mean
sst_ML_ly1 = sst_ML_ly1.mean(dim='sfc')

In [ ]:
# Load, slice, and collect datasets
sliced_datasets1 = []
sliced_datasets2 = []
for y_id in range(time_min,time_max):
    path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/tas_ML_NA/anomaly/'
    file = 'tas_%i_r1-16i2p3-LR_26_months_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path+file) as tas_ML:
        tas_ML_ly1 = tas_ML.tas.isel(time=slice(lead_min,lead_max))
    sliced_datasets1.append(tas_ML_ly1.mean(dim='time').expand_dims(predicted_year=[y_id]))
    
#concatenate along time dimension
tas_ML_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
tas_ML_ly1['lon'] = np.where(tas_ML_ly1.lon > 180,tas_ML_ly1.lon-360,tas_ML_ly1.lon)
ind_sort = np.argsort(tas_ML_ly1.lon)
tas_ML_ly1 = tas_ML_ly1[:,:,:,ind_sort.values]

#give different coordinates for each ensemble member in sfc:
tas_ML_ly1 = tas_ML_ly1.assign_coords(sfc=np.arange(1, 17))

#ensemble mean
tas_ML_ly1 = tas_ML_ly1.mean(dim='sfc')

# Standard

In [ ]:
# Load, slice, and collect datasets
sliced_datasets1 = []
sliced_datasets2 = []
time_info1 = []
time_info2 = []
for y_id in range(time_min,time_max):
    path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/sst_benchmark/anomaly/'
    file = 'sst_%i_r1-16i2p2-LR_3_years_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path+file) as sst_MPI:
        sst_MPI_ly1 = sst_MPI.tos.isel(time=slice(lead_min,lead_max))
        time_info1.append((sst_MPI_ly1.time.min(),sst_MPI_ly1.time.max()))      
    sliced_datasets1.append(sst_MPI_ly1.mean(dim='time').expand_dims(predicted_year=[y_id]))
    
#concatenate along time dimension
sst_MPI_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
sst_MPI_ly1['lon'] = np.where(sst_MPI_ly1.lon > 180,sst_MPI_ly1.lon-360,sst_MPI_ly1.lon)
ind_sort = np.argsort(sst_MPI_ly1.lon)
sst_MPI_ly1 = sst_MPI_ly1[:,:,:,ind_sort.values]

#change 'depth' dimension name to 'sfc', so the dimension name matches with z500 (only for ML, not for MPI)
sst_MPI_ly1 = sst_MPI_ly1.rename({"depth": "sfc"})
#give different coordinates for each ensemble member in sfc:
sst_MPI_ly1 = sst_MPI_ly1.assign_coords(sfc=np.arange(1, 17))

#ensemble mean
sst_MPI_ly1 = sst_MPI_ly1.mean(dim='sfc')

In [ ]:
# Load, slice, and collect datasets
sliced_datasets1 = []
sliced_datasets2 = []
time_info1 = []
time_info2 = []
for y_id in range(time_min,time_max):
    path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/tas_benchmark/anomaly/'
    file = 'tas_%i_r1-16i2p2-LR_3_years_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path+file) as tas_MPI:
        tas_MPI_ly1 = tas_MPI.tas.isel(time=slice(lead_min,lead_max))
        time_info1.append((tas_MPI_ly1.time.min(),tas_MPI_ly1.time.max()))
    sliced_datasets1.append(tas_MPI_ly1.mean(dim='time').expand_dims(predicted_year=[y_id]))
    
#concatenate along time dimension
tas_MPI_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
tas_MPI_ly1['lon'] = np.where(tas_MPI_ly1.lon > 180,tas_MPI_ly1.lon-360,tas_MPI_ly1.lon)
ind_sort = np.argsort(tas_MPI_ly1.lon)
tas_MPI_ly1 = tas_MPI_ly1[:,:,:,ind_sort.values]

#give different coordinates for each ensemble member in sfc:
tas_MPI_ly1 = tas_MPI_ly1.assign_coords(sfc=np.arange(1, 17))

#ensemble mean
tas_MPI_ly1 = tas_MPI_ly1.mean(dim='sfc')

# ERA5

In [ ]:
#ERA
path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/era5/'
file = 'era5_sst_1940-2023_360x180_anomaly_ref_%i-%i.nc' %(ref_min,ref_max)
sst_era = xr.open_dataset(path+file).sst

#new longitudes
sst_era['lon'] = np.where(sst_era.lon > 180,sst_era.lon-360,sst_era.lon)
ind_sort = np.argsort(sst_era.lon)
sst_era = sst_era[:,:,ind_sort.values]

#homogenize to MPI and ML
sliced_datasets1 = []
for y_id,year in enumerate(np.arange(time_min,time_max)):
    sst_era_ly1 = sst_era[(sst_era.time.dt.floor('D')>=time_info1[y_id][0].dt.floor('D'))&(sst_era.time.dt.floor('D')<=time_info1[y_id][1].dt.floor('D'))]
    sliced_datasets1.append(sst_era_ly1.mean(dim='time').expand_dims(predicted_year=[year]))
    
#concatenate along time dimension
sst_era_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
sst_era_ly1['lon'] = np.where(sst_era_ly1.lon > 180,sst_era_ly1.lon-360,sst_era_ly1.lon)
ind_sort = np.argsort(sst_era_ly1.lon)
sst_era_ly1 = sst_era_ly1[:,:,ind_sort.values]

In [ ]:
#ERA
path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/era5/'
file = 'era5_tas_1940-2023_360x180_anomaly_ref_%i-%i.nc' %(ref_min,ref_max)
tas_era = xr.open_dataset(path+file).t2m[:,0] #remove exp variable

#new longitudes
tas_era['lon'] = np.where(tas_era.lon > 180,tas_era.lon-360,tas_era.lon)
ind_sort = np.argsort(tas_era.lon)
tas_era = tas_era[:,:,ind_sort.values]

#homogenize to MPI and ML
sliced_datasets1 = []
sliced_datasets2 = []
for y_id,year in enumerate(np.arange(time_min,time_max)):
    tas_era_ly1 = tas_era[(tas_era.time.dt.floor('D')>=time_info1[y_id][0].dt.floor('D'))&(tas_era.time.dt.floor('D')<=time_info1[y_id][1].dt.floor('D'))]
    sliced_datasets1.append(tas_era_ly1.mean(dim='time').expand_dims(predicted_year=[year]))
    
#concatenate along time dimension
tas_era_ly1 = xr.concat(sliced_datasets1,dim='predicted_year')

#new longitudes
tas_era_ly1['lon'] = np.where(tas_era_ly1.lon > 180,tas_era_ly1.lon-360,tas_era_ly1.lon)
ind_sort = np.argsort(tas_era_ly1.lon)
tas_era_ly1 = tas_era_ly1[:,:,ind_sort.values]

# Lead year correlation

In [ ]:
def pearsonr_func(x, y):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    # mask NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(mask) < 2:
        return np.nan, np.nan

    r, p = pearsonr(x[mask], y[mask])
    return r,p

In [ ]:
def compute_correlation(x_corr,y_corr):
    r, p = xr.apply_ufunc(
        pearsonr_func,
        x_corr, y_corr,
        input_core_dims=[["predicted_year"], ["predicted_year"]],
        output_core_dims=[[], []],      # r and p are scalars per gridcell
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float, float]
    )
    return r,p

In [ ]:
period_min1 = 1965
period_max1 = 1994
period_min2 = 1991
period_max2 = 2020

In [ ]:
#SST
#MPI
x_corr = sst_era_ly1.sel(predicted_year=slice(period_min1,period_max1))
y_corr = sst_MPI_ly1.sel(predicted_year=slice(period_min1,period_max1))
r_sst_MPI_ly1_per1, p_sst_MPI_ly1_per1 = compute_correlation(x_corr,y_corr)

x_corr = sst_era_ly1.sel(predicted_year=slice(period_min2,period_max2))
y_corr = sst_MPI_ly1.sel(predicted_year=slice(period_min2,period_max2))
r_sst_MPI_ly1_per2, p_sst_MPI_ly1_per2 = compute_correlation(x_corr,y_corr)

#ML
x_corr = sst_era_ly1.sel(predicted_year=slice(period_min1,period_max1))
y_corr = sst_ML_ly1.sel(predicted_year=slice(period_min1,period_max1))
r_sst_ML_ly1_per1, p_sst_ML_ly1_per1 = compute_correlation(x_corr,y_corr)

x_corr = sst_era_ly1.sel(predicted_year=slice(period_min2,period_max2))
y_corr = sst_ML_ly1.sel(predicted_year=slice(period_min2,period_max2))
r_sst_ML_ly1_per2, p_sst_ML_ly1_per2 = compute_correlation(x_corr,y_corr)

In [ ]:
#Tas
#MPI
x_corr = tas_era_ly1.sel(predicted_year=slice(period_min1,period_max1))
y_corr = tas_MPI_ly1.sel(predicted_year=slice(period_min1,period_max1))
r_tas_MPI_ly1_per1, p_tas_MPI_ly1_per1 = compute_correlation(x_corr,y_corr)

x_corr = tas_era_ly1.sel(predicted_year=slice(period_min2,period_max2))
y_corr = tas_MPI_ly1.sel(predicted_year=slice(period_min2,period_max2))
r_tas_MPI_ly1_per2, p_tas_MPI_ly1_per2 = compute_correlation(x_corr,y_corr)

#ML
x_corr = tas_era_ly1.sel(predicted_year=slice(period_min1,period_max1))
y_corr = tas_ML_ly1.sel(predicted_year=slice(period_min1,period_max1))
r_tas_ML_ly1_per1, p_tas_ML_ly1_per1 = compute_correlation(x_corr,y_corr)

x_corr = tas_era_ly1.sel(predicted_year=slice(period_min2,period_max2))
y_corr = tas_ML_ly1.sel(predicted_year=slice(period_min2,period_max2))
r_tas_ML_ly1_per2, p_tas_ML_ly1_per2 = compute_correlation(x_corr,y_corr)

In [ ]:
#Inser tas values over land areas
r_ML_ly1_per1 = xr.where(np.isnan(r_sst_ML_ly1_per1),r_tas_ML_ly1_per1,r_sst_ML_ly1_per1)
r_ML_ly1_per2 = xr.where(np.isnan(r_sst_ML_ly1_per2),r_tas_ML_ly1_per2,r_sst_ML_ly1_per2)

p_ML_ly1_per1 = xr.where(np.isnan(p_sst_ML_ly1_per1),p_tas_ML_ly1_per1,p_sst_ML_ly1_per1)
p_ML_ly1_per2 = xr.where(np.isnan(p_sst_ML_ly1_per2),p_tas_ML_ly1_per2,p_sst_ML_ly1_per2)

In [ ]:
#Inser tas values over land areas
r_MPI_ly1_per1 = xr.where(np.isnan(r_sst_MPI_ly1_per1),r_tas_MPI_ly1_per1,r_sst_MPI_ly1_per1)
r_MPI_ly1_per2 = xr.where(np.isnan(r_sst_MPI_ly1_per2),r_tas_MPI_ly1_per2,r_sst_MPI_ly1_per2)

p_MPI_ly1_per1 = xr.where(np.isnan(p_sst_MPI_ly1_per1),p_tas_MPI_ly1_per1,p_sst_MPI_ly1_per1)
p_MPI_ly1_per2 = xr.where(np.isnan(p_sst_MPI_ly1_per2),p_tas_MPI_ly1_per2,p_sst_MPI_ly1_per2)

# Running window skill for each domain

In [ ]:
lon_min = -100
lon_max = 40
lat_min = 0
lat_max = 80

#select SPG area
lon_min_spg = -60
lon_max_spg = -10
lat_min_spg = 50
lat_max_spg = 66

#select CE area
lon_min_ce = 5
lon_max_ce = 30#10
lat_min_ce = 48
lat_max_ce = 65#58

#select North America area
lon_min_nam = -77
lon_max_nam = -57
lat_min_nam = 47
lat_max_nam = 61

#select Greenland area
lon_min_gl = -55
lon_max_gl = -35
lat_min_gl = 60
lat_max_gl = 75
#joker

In [ ]:
period_min = 1960
period_max = 1991
window = 30

r_spg_ML_ly1_list = [] 
p_spg_ML_ly1_list = []
r_ce_ML_ly1_list = [] 
p_ce_ML_ly1_list = []
r_nam_ML_ly1_list = [] 
p_nam_ML_ly1_list = []
r_gl_ML_ly1_list = [] 
p_gl_ML_ly1_list = []
r_spg_MPI_ly1_list = [] 
p_spg_MPI_ly1_list = []
r_ce_MPI_ly1_list = [] 
p_ce_MPI_ly1_list = []
r_nam_MPI_ly1_list = [] 
p_nam_MPI_ly1_list = []
r_gl_MPI_ly1_list = [] 
p_gl_MPI_ly1_list = []

for y in range(period_min,period_max+1):
    w_start = y
    w_end = y+window-1

    #ERA
    sel_lon = (tas_era_ly1.lon>= lon_min_ce) & (tas_era_ly1.lon <= lon_max_ce)
    sel_lat = (tas_era_ly1.lat>= lat_min_ce) & (tas_era_ly1.lat <= lat_max_ce)
    tas_era_ly1_ce = tas_era_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_era_ly1.lon>= lon_min_nam) & (tas_era_ly1.lon <= lon_max_nam)
    sel_lat = (tas_era_ly1.lat>= lat_min_nam) & (tas_era_ly1.lat <= lat_max_nam)
    tas_era_ly1_nam = tas_era_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_era_ly1.lon>= lon_min_gl) & (tas_era_ly1.lon <= lon_max_gl)
    sel_lat = (tas_era_ly1.lat>= lat_min_gl) & (tas_era_ly1.lat <= lat_max_gl)
    tas_era_ly1_gl = tas_era_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (sst_era_ly1.lon>= lon_min_spg) & (sst_era_ly1.lon <= lon_max_spg)
    sel_lat = (sst_era_ly1.lat>= lat_min_spg) & (sst_era_ly1.lat <= lat_max_spg)
    sst_era_ly1_spg = sst_era_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    #MPI
    sel_lon = (tas_MPI_ly1.lon>= lon_min_ce) & (tas_MPI_ly1.lon <= lon_max_ce)
    sel_lat = (tas_MPI_ly1.lat>= lat_min_ce) & (tas_MPI_ly1.lat <= lat_max_ce)
    tas_MPI_ly1_ce = tas_MPI_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_MPI_ly1.lon>= lon_min_nam) & (tas_MPI_ly1.lon <= lon_max_nam)
    sel_lat = (tas_MPI_ly1.lat>= lat_min_nam) & (tas_MPI_ly1.lat <= lat_max_nam)
    tas_MPI_ly1_nam = tas_MPI_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_MPI_ly1.lon>= lon_min_gl) & (tas_MPI_ly1.lon <= lon_max_gl)
    sel_lat = (tas_MPI_ly1.lat>= lat_min_gl) & (tas_MPI_ly1.lat <= lat_max_gl)
    tas_MPI_ly1_gl = tas_MPI_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (sst_MPI_ly1.lon>= lon_min_spg) & (sst_MPI_ly1.lon <= lon_max_spg)
    sel_lat = (sst_MPI_ly1.lat>= lat_min_spg) & (sst_MPI_ly1.lat <= lat_max_spg)
    sst_MPI_ly1_spg = sst_MPI_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    #ML
    sel_lon = (tas_ML_ly1.lon>= lon_min_ce) & (tas_ML_ly1.lon <= lon_max_ce)
    sel_lat = (tas_ML_ly1.lat>= lat_min_ce) & (tas_ML_ly1.lat <= lat_max_ce)
    tas_ML_ly1_ce = tas_ML_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_ML_ly1.lon>= lon_min_nam) & (tas_ML_ly1.lon <= lon_max_nam)
    sel_lat = (tas_ML_ly1.lat>= lat_min_nam) & (tas_ML_ly1.lat <= lat_max_nam)
    tas_ML_ly1_nam = tas_ML_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (tas_ML_ly1.lon>= lon_min_gl) & (tas_ML_ly1.lon <= lon_max_gl)
    sel_lat = (tas_ML_ly1.lat>= lat_min_gl) & (tas_ML_ly1.lat <= lat_max_gl)
    tas_ML_ly1_gl = tas_ML_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))
    
    sel_lon = (sst_ML_ly1.lon>= lon_min_spg) & (sst_ML_ly1.lon <= lon_max_spg)
    sel_lat = (sst_ML_ly1.lat>= lat_min_spg) & (sst_ML_ly1.lat <= lat_max_spg)
    sst_ML_ly1_spg = sst_ML_ly1[:,sel_lat,sel_lon].mean(dim=('lon','lat'))

    #SPG
    #MPI
    x_corr = sst_era_ly1_spg.sel(predicted_year=slice(w_start,w_end))
    y_corr = sst_MPI_ly1_spg.sel(predicted_year=slice(w_start,w_end))
    r_spg_MPI_ly1_per, p_spg_MPI_ly1_per = pearsonr(x_corr,y_corr)
    
    #ML
    x_corr = sst_era_ly1_spg.sel(predicted_year=slice(w_start,w_end))
    y_corr = sst_ML_ly1_spg.sel(predicted_year=slice(w_start,w_end))
    r_spg_ML_ly1_per, p_spg_ML_ly1_per = compute_correlation(x_corr,y_corr)
    
    #CE Tas
    #MPI
    x_corr = tas_era_ly1_ce.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_MPI_ly1_ce.sel(predicted_year=slice(w_start,w_end))
    r_ce_MPI_ly1_per, p_ce_MPI_ly1_per = compute_correlation(x_corr,y_corr)
    
    #ML
    x_corr = tas_era_ly1_ce.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_ML_ly1_ce.sel(predicted_year=slice(w_start,w_end))
    r_ce_ML_ly1_per, p_ce_ML_ly1_per = compute_correlation(x_corr,y_corr)
    
    #NAM Tas
    #MPI
    x_corr = tas_era_ly1_nam.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_MPI_ly1_nam.sel(predicted_year=slice(w_start,w_end))
    r_nam_MPI_ly1_per, p_nam_MPI_ly1_per = compute_correlation(x_corr,y_corr)
    
    #ML
    x_corr = tas_era_ly1_nam.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_ML_ly1_nam.sel(predicted_year=slice(w_start,w_end))
    r_nam_ML_ly1_per, p_nam_ML_ly1_per = compute_correlation(x_corr,y_corr)
    
    #GL Tas
    #MPI
    x_corr = tas_era_ly1_gl.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_MPI_ly1_gl.sel(predicted_year=slice(w_start,w_end))
    r_gl_MPI_ly1_per, p_gl_MPI_ly1_per = compute_correlation(x_corr,y_corr)
    
    #ML
    x_corr = tas_era_ly1_gl.sel(predicted_year=slice(w_start,w_end))
    y_corr = tas_ML_ly1_gl.sel(predicted_year=slice(w_start,w_end))
    r_gl_ML_ly1_per, p_gl_ML_ly1_per = compute_correlation(x_corr,y_corr)

    #APPEND
    r_spg_ML_ly1_list.append(r_spg_ML_ly1_per)
    p_spg_ML_ly1_list.append(p_spg_ML_ly1_per)
    r_ce_ML_ly1_list.append(r_ce_ML_ly1_per)
    p_ce_ML_ly1_list.append(p_ce_ML_ly1_per)
    r_nam_ML_ly1_list.append(r_nam_ML_ly1_per)
    p_nam_ML_ly1_list.append(p_nam_ML_ly1_per)
    r_gl_ML_ly1_list.append(r_gl_ML_ly1_per)
    p_gl_ML_ly1_list.append(p_gl_ML_ly1_per)

    r_spg_MPI_ly1_list.append(r_spg_MPI_ly1_per)
    p_spg_MPI_ly1_list.append(p_spg_MPI_ly1_per)
    r_ce_MPI_ly1_list.append(r_ce_MPI_ly1_per)
    p_ce_MPI_ly1_list.append(p_ce_MPI_ly1_per)
    r_nam_MPI_ly1_list.append(r_nam_MPI_ly1_per)
    p_nam_MPI_ly1_list.append(p_nam_MPI_ly1_per)
    r_gl_MPI_ly1_list.append(r_gl_MPI_ly1_per)
    p_gl_MPI_ly1_list.append(p_gl_MPI_ly1_per)

    print('Window starting on %i done' %w_start)

# Plot with gray scale for non-significant correlations

In [ ]:
#check for the limit of significance in ML
plot_data = r_ML_ly1_per1
pvals = p_ML_ly1_per1

sig = np.ma.masked_where(pvals > 0.05, plot_data)
nosig = np.ma.masked_where(pvals <= 0.05, plot_data)

print(sig.min())
print(nosig.min())

In [ ]:
#check for the limit of significance in MPI
plot_data = r_MPI_ly1_per1
pvals = p_MPI_ly1_per1

sig = np.ma.masked_where(pvals > 0.05, plot_data)
nosig = np.ma.masked_where(pvals <= 0.05, plot_data)

print(sig.min())
print(nosig.max())

In [ ]:
#Find blue to red colors
# Original colormap
cmap = plt.cm.RdBu_r
cmaplist = [cmap(i) for i in range(cmap.N)]
cmap = mpl.colors.LinearSegmentedColormap.from_list('Custom cmap', cmaplist, cmap.N)

# Discretization bounds
bounds = np.linspace(-1, 1, 11)  # 10 intervals
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

# Get midpoint of each interval
midpoints = 0.5*(bounds[:-1] + bounds[1:])

# Convert midpoints to 0-1 range for the colormap
colors = [cmap((mp - bounds[0]) / (bounds[-1] - bounds[0])) for mp in midpoints]

# Print RGBA colors for each range
for i, (low, high) in enumerate(zip(bounds[:-1], bounds[1:])):
    print(f"Range {low:.2f} to {high:.2f} -> Color RGBA {colors[i]}")

In [ ]:
import matplotlib as mpl

# Define bounds for discretization
bounds = np.linspace(-1, 1, 11)  # 10 intervals
bounds_diff = np.linspace(-0.5, 0.5, 11)  # 10 intervals

#set manually the interval for significance: here is 0.35
bounds[3]=-0.35
bounds[7]= 0.35

# Manually define a color for each interval (10 colors)
# Example: 3 blues, 4 grays, 3 reds
colors_combined = [
    colors[0],   # dark blue
    colors[1],   # medium blue
    colors[2],   # light blue
    (0.8, 0.8, 0.8, 1.0),   # gray
    (1.0, 1.0, 1.0, 1.0),   # white
    (1.0, 1.0, 1.0, 1.0),   # white
    (0.8, 0.8, 0.8, 1.0),   # gray
    colors[-3],   # light red
    colors[-2],   # medium red
    colors[-1]    # dark red
]

colors_diff = [
    colors[0],   # dark blue
    colors[1],   # medium blue
    colors[2],   # light blue
    colors[3],
    colors[4],
    colors[5],
    colors[6],
    colors[-3],   # light red
    colors[-2],   # medium red
    colors[-1]    # dark red
]

# Create discrete colormap
cmap_combined = mpl.colors.ListedColormap(colors_combined)
cmap_diff = mpl.colors.ListedColormap(colors_diff)

# Create corresponding norm
norm_combined = mpl.colors.BoundaryNorm(bounds, cmap_combined.N)
norm_diff = mpl.colors.BoundaryNorm(bounds_diff, cmap_diff.N)

In [ ]:
lon_min = -80
lon_max = 35
lat_min = 30
lat_max = 80

In [ ]:
# -----------------------------
# Figure setup
# -----------------------------
fig = plt.figure(figsize=(15,10))
plt.rcParams.update({'font.size': 12})
#gs = GridSpec(10, 10, height_ratios=[1,1], figure=fig)
gs = GridSpec(11, 11, figure=fig, wspace=0.6, hspace=0.1)

x_plot = np.arange(period_min+window-1,period_max+window)
# Labels showing the full period represented by each point
tick_years = np.array([1990, 2000, 2010, 2020])
x_labels = [
    f"{year - window + 1}\u2013\n{year}"
    for year in tick_years
]

# =============================
# PANEL A: ML
# =============================
plot_data = r_ML_ly1_per1
pvals = p_ML_ly1_per1
ax1 = fig.add_subplot(gs[0:3,0:5], projection=ccrs.PlateCarree())

pcm_color = ax1.pcolormesh(plot_data.lon, plot_data.lat, plot_data,
                           cmap=cmap_combined, norm=norm_combined,
                           transform=ccrs.PlateCarree(), zorder=2)

ax1.coastlines(resolution='110m', lw=0.5, color='k', zorder=3)
ax1.set_facecolor('w')
plt.xlim([lon_min, lon_max])
plt.ylim([lat_min, lat_max])

# Colorbars
divider = make_axes_locatable(ax1)
cax_color = divider.append_axes("right", size="5%", pad=0.25, axes_class=maxes.Axes)
#cax_color.set_visible(False)
selected_ticks = [-1, -0.8, -0.6,-0.35,-0.2, 0, 0.2,0.35,0.6, 0.8,1]
plt.colorbar(pcm_color, cax=cax_color, ticks=selected_ticks)

ax1.text(0.5, 1.05, 'Hybrid-ML', transform=ax1.transAxes, ha='center', va='bottom', fontsize=13, fontweight='bold')
ax1.text(-0.07,1.07,'a)', transform=ax1.transAxes, ha='left', va='top', fontsize=14, fontweight='bold')

# =============================
# PANEL B: MPI
# =============================
plot_data = r_MPI_ly1_per1
pvals = p_MPI_ly1_per1
ax2 = fig.add_subplot(gs[0:3,6:11], projection=ccrs.PlateCarree())

pcm_color = ax2.pcolormesh(plot_data.lon, plot_data.lat, plot_data,
                           cmap=cmap_combined, norm=norm_combined,
                           transform=ccrs.PlateCarree(), zorder=2)

ax2.coastlines(resolution='110m', lw=0.5, color='k', zorder=3)
ax2.set_facecolor('w')
plt.xlim([lon_min, lon_max])
plt.ylim([lat_min, lat_max])

divider = make_axes_locatable(ax2)
cax_color = divider.append_axes("right", size="5%", pad=0.25, axes_class=maxes.Axes)
selected_ticks = [-1, -0.8, -0.6,-0.35,-0.2, 0, 0.2,0.35,0.6, 0.8,1]
plt.colorbar(pcm_color, cax=cax_color, ticks=selected_ticks)

ax2.text(0.5, 1.05, 'Standard', transform=ax2.transAxes, ha='center', va='bottom', fontsize=13, fontweight='bold')
ax2.text(-0.07,1.07,'b)', transform=ax2.transAxes, ha='left', va='top', fontsize=14, fontweight='bold')

# =============================
# PANEL C: ML-MPI
# =============================
plot_data = r_ML_ly1_per1 - r_MPI_ly1_per1
ax3 = fig.add_subplot(gs[5:8,3:8], projection=ccrs.PlateCarree())

pcm = ax3.pcolormesh(plot_data.lon, plot_data.lat, plot_data,
                     cmap=cmap_diff, norm=norm_diff,
                     transform=ccrs.PlateCarree())
ax3.coastlines(resolution='110m', lw=0.5, color='k')
ax3.set_facecolor('w')
plt.xlim([lon_min, lon_max])
plt.ylim([lat_min, lat_max])


cax = inset_axes(ax3, width="90%", height="8%", loc="lower center",
                 bbox_to_anchor=(0, -0.15, 1, 1),
                 bbox_transform=ax3.transAxes, borderpad=0)
selected_ticks = [-0.5, -0.4, -0.3,-0.2,-0.1, 0, 0.1,0.2,0.3, 0.4,0.5]
cb = plt.colorbar(pcm, cax=cax, ticks=selected_ticks,orientation='horizontal')
cb.ax.set_xticklabels(cb.ax.get_xticklabels(), rotation=25, ha='right')

ax3.text(0.5, 1.05, 'Hybrid-ML - Standard', transform=ax3.transAxes, ha='center', va='bottom', fontsize=13, fontweight='bold')
ax3.text(-0.07,1.07,'c)', transform=ax3.transAxes, ha='left', va='top', fontsize=14, fontweight='bold')

# -----------------------------
# Draw subregion rectangles on ML-MPI panel
# -----------------------------
region_color = 'dimgray'#"#333333"
for lon0, lat0, lon1, lat1 in [
    (lon_min_spg, lat_min_spg, lon_max_spg, lat_max_spg),
    (lon_min_ce, lat_min_ce, lon_max_ce, lat_max_ce),
    (lon_min_nam, lat_min_nam, lon_max_nam, lat_max_nam),
    (lon_min_gl, lat_min_gl, lon_max_gl, lat_max_gl)]:
    ax3.add_patch(Rectangle((lon0, lat0), lon1-lon0, lat1-lat0,
                            linewidth=1.5, edgecolor=region_color, facecolor='none', transform=ccrs.PlateCarree()))

ax3.text(
    lon_max_spg, lat_max_spg, 'c1',   # position (x,y)
    transform=ax3.transData,
    ha='right', va='bottom',
    fontsize=12, fontweight='bold', color=region_color)

ax3.text(
    lon_min_gl, lat_max_gl, 'c2',   # position (x,y)
    transform=ax3.transData,
    ha='left', va='bottom',
    fontsize=12, fontweight='bold', color=region_color)

ax3.text(
    lon_max_ce, lat_max_ce, 'c4',   # position (x,y)
    transform=ax3.transData,
    ha='right', va='bottom',
    fontsize=12, fontweight='bold', color=region_color)

ax3.text(
    lon_min_nam, lat_max_nam, 'c3',   # position (x,y)
    transform=ax3.transData,
    ha='left', va='bottom',
    fontsize=12, fontweight='bold', color=region_color)


# -----------------------------
# Plot skill for subregions
# -----------------------------
ylim_min=-0.05
ylim_max=0.95

if lead_year == 2:
    ylim_min = -0.2

# =========== GL ============
ax4= fig.add_subplot(gs[4:6,8:11])
plt.plot(x_plot,r_gl_ML_ly1_list,'#1b9e77',label='Hybrid-ML')
plt.plot(x_plot,r_gl_MPI_ly1_list,'#7570b3',label='Standard')
# add X on significant skill
sig_ML = np.array(p_gl_ML_ly1_list) < 0.05
sig_MPI = np.array(p_gl_MPI_ly1_list) < 0.05
ax4.scatter(x_plot[sig_MPI],
            np.array(r_gl_MPI_ly1_list)[sig_MPI],
            marker='x', color='#7570b3', s=50, label='_nolegend_')

ax4.scatter(x_plot[sig_ML],
            np.array(r_gl_ML_ly1_list)[sig_ML],
            marker='x', color='#1b9e77', s=50, label='_nolegend_')
plt.legend(loc='lower right',fontsize=10)
plt.ylim([ylim_min,ylim_max])
plt.grid()

ax4.text(0.5, 1.05, 'Southern Greenland', transform=ax4.transAxes, ha='center', va='bottom', fontsize=12, fontweight='bold',color=region_color)
ax4.text(-0.1,1.12,'c2)', transform=ax4.transAxes, ha='left', va='top', fontsize=11, fontweight='bold',color=region_color)
ax4.set_xticks(tick_years)
ax4.set_xticklabels(x_labels, ha='center', fontsize=10)


# =========== NAM ============
ax5= fig.add_subplot(gs[7:9,0:3])
plt.plot(x_plot,r_nam_ML_ly1_list,'#1b9e77',label='Hybrid-ML')
plt.plot(x_plot,r_nam_MPI_ly1_list,'#7570b3',label='Standard')
# add X on significant skill
sig_ML = np.array(p_nam_ML_ly1_list) < 0.05
sig_MPI = np.array(p_nam_MPI_ly1_list) < 0.05
ax5.scatter(x_plot[sig_MPI],
            np.array(r_nam_MPI_ly1_list)[sig_MPI],
            marker='x', color='#7570b3', s=50, label='_nolegend_')

ax5.scatter(x_plot[sig_ML],
            np.array(r_nam_ML_ly1_list)[sig_ML],
            marker='x', color='#1b9e77', s=50, label='_nolegend_')
plt.legend(loc='lower right',fontsize=10)
plt.ylim([ylim_min,ylim_max])
plt.grid()
ax5.text(0.5, 1.05, 'Northeastern America', transform=ax5.transAxes, ha='center', va='bottom', fontsize=12, fontweight='bold',color=region_color)
ax5.text(-0.1,1.12,'c3)', transform=ax5.transAxes, ha='left', va='top', fontsize=11, fontweight='bold',color=region_color)
ax5.set_xticks(tick_years)
ax5.set_xticklabels(x_labels, ha='center', fontsize=10)

# =========== SPG ============
ax6= fig.add_subplot(gs[4:6,0:3])
plt.plot(x_plot,r_spg_ML_ly1_list,'#1b9e77',label='Hybrid-ML')
plt.plot(x_plot,r_spg_MPI_ly1_list,'#7570b3',label='Standard')
# add X on significant skill
sig_ML = np.array(p_spg_ML_ly1_list) < 0.05
sig_MPI = np.array(p_spg_MPI_ly1_list) < 0.05
ax6.scatter(x_plot[sig_MPI],
            np.array(r_spg_MPI_ly1_list)[sig_MPI],
            marker='x', color='#7570b3', s=50, label='_nolegend_')

ax6.scatter(x_plot[sig_ML],
            np.array(r_spg_ML_ly1_list)[sig_ML],
            marker='x', color='#1b9e77', s=50, label='_nolegend_')
plt.legend(loc='lower right',fontsize=10)
plt.ylim([ylim_min,ylim_max])
plt.grid()
ax6.text(0.5, 1.05, 'SPG', transform=ax6.transAxes, ha='center', va='bottom', fontsize=12, fontweight='bold',color=region_color)
ax6.text(-0.1,1.12,'c1)', transform=ax6.transAxes, ha='left', va='top', fontsize=11, fontweight='bold',color=region_color)
ax6.set_xticks(tick_years)
ax6.set_xticklabels(x_labels, ha='center', fontsize=10)

# =========== CE ============
ax7= fig.add_subplot(gs[7:9,8:11])
plt.plot(x_plot,r_ce_ML_ly1_list,'#1b9e77',label='Hybrid-ML')
plt.plot(x_plot,r_ce_MPI_ly1_list,'#7570b3',label='Standard')
# add X on significant skill
sig_ML = np.array(p_ce_ML_ly1_list) < 0.05
sig_MPI = np.array(p_ce_MPI_ly1_list) < 0.05
ax7.scatter(x_plot[sig_MPI],
            np.array(r_ce_MPI_ly1_list)[sig_MPI],
            marker='x', color='#7570b3', s=50, label='_nolegend_')

ax7.scatter(x_plot[sig_ML],
            np.array(r_ce_ML_ly1_list)[sig_ML],
            marker='x', color='#1b9e77', s=50, label='_nolegend_')
plt.legend(loc='lower right',fontsize=10)
plt.ylim([ylim_min,ylim_max])
plt.grid()
ax7.text(0.5, 1.05, 'Central-Northern Europe', transform=ax7.transAxes, ha='center', va='bottom', fontsize=12, fontweight='bold',color=region_color)
ax7.text(-0.1,1.12,'c4)', transform=ax7.transAxes, ha='left', va='top', fontsize=11, fontweight='bold',color=region_color)
ax7.set_xticks(tick_years)
ax7.set_xticklabels(x_labels, ha='center', fontsize=10)

# -----------------------------
# Save figure
# -----------------------------
plt.savefig(plot_path+'sst_tas_annual_skill_%ileadyear_period_%i-%i_anomaly_ref_%i-%i'%(
            lead_year,period_min1,period_max1,ref_min,ref_max), bbox_inches="tight", dpi=100)